# 01 · Der eigene Produkt-Retriever – Colab-Fassung
## mLateOn für Produktsuche finetunen, mit Amazon ESCI

**Motivation aus einem Kundenprojekt:** Rund eine halbe Million Produkte sollen öffentlichen Ausschreibungen
zugeordnet werden. Dense Retrieval und BM25 finden dort nur einen kleinen Bruchteil der relevanten Produkte,
Hybrid mit Augmentation deutlich mehr, ein speziell trainiertes ColBERT noch einmal klar mehr – nicht perfekt,
aber immerhin. Dieses Notebook zeigt dieselbe Technik mit öffentlichen Daten:
[Amazon ESCI](https://github.com/amazon-science/esci-data) – echte Shopping-Queries mit menschlichen Urteilen.

**Ablauf:** Pakete → ESCI laden → Trainingstripel mit Hard Negatives → Full-Fine-Tuning von
[`lightonai/mLateOn-unsupervised`](https://huggingface.co/lightonai/mLateOn-unsupervised) mit Checkpoints und
Dev-Evaluation → speichern und neu laden → exakte Retrieval-Evaluation gegen BM25, das Basismodell und das
veröffentlichte Vollmodell [`johannhartmann/mlateon-esci-product-search`](https://huggingface.co/johannhartmann/mlateon-esci-product-search)
→ LLM-vervollständigte Metriken → Suche vorher/nachher → Export (Hub, Drive).

**Start:** Laufzeit → Laufzeittyp ändern → GPU (T4, L4 oder A100), dann *Laufzeit → Alle ausführen*.
Das Notebook erkennt GPU, RAM und Platte und wählt das passende Profil. Alles, was es braucht, lädt es selbst:
Pakete von PyPI, ESCI von GitHub, Modelle, LLM-Urteile und das Auswertungsmodul vom Hugging Face Hub.

## 1 · Pakete

Sentence Transformers 6.1.0 bringt `MultiVectorEncoder` (ColBERT/Late Interaction) mit Trainer, Losses und
Evaluatoren. Dazu `bm25s` für die BM25-Baseline. PyTorch, NumPy und ipywidgets der Colab-Laufzeit bleiben
unverändert. Meldungen des pip-Resolvers über andere vorinstallierte Colab-Pakete betreffen dieses Notebook nicht.
Waren betroffene Bibliotheken schon importiert, startet die Zelle die Laufzeit neu – danach erneut *Alle ausführen*.

In [ ]:
# @title Pakete installieren
import importlib.metadata as metadata
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PINNED = {"sentence-transformers": "6.1.0", "transformers": "5.17.0", "bm25s": "0.3.11", "PyStemmer": "3.1.0"}


def installed(distribution):
    try:
        return metadata.version(distribution)
    except metadata.PackageNotFoundError:
        return None


if any(installed(d) != v for d, v in PINNED.items()) or not (installed("accelerate") and installed("datasets")):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        f"sentence-transformers[train]=={PINNED['sentence-transformers']}",
        *(f"{d}=={v}" for d, v in PINNED.items() if d != "sentence-transformers"),
    ])

# Schon importierte Module in alter Version wären inkonsistent: dann Neustart.
MODULES = {"transformers": "transformers", "sentence_transformers": "sentence-transformers",
           "huggingface_hub": "huggingface-hub", "tokenizers": "tokenizers", "datasets": "datasets",
           "accelerate": "accelerate"}
stale = [m for m, d in MODULES.items()
         if m in sys.modules and getattr(sys.modules[m], "__version__", None) != installed(d)]
if stale:
    if IN_COLAB:
        print("Neue Versionen installiert, alte schon geladen:", ", ".join(stale))
        print("Laufzeit startet neu – danach erneut 'Alle ausführen'.")
        os.kill(os.getpid(), 9)
    raise RuntimeError(f"Bereits geladen in alter Version: {stale}. Kernel neu starten und erneut ausführen.")
print(" · ".join(f"{d} {installed(d)}" for d in
                 ["torch", "numpy", "sentence-transformers", "transformers", "datasets", "huggingface-hub", "bm25s"]))

In [ ]:
# @title Bibliotheken, Hardware, Hilfsmodul
import gc
import hashlib
import html
import json
import math
import random
import re
import resource
import shutil
import time
from contextlib import contextmanager
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import requests
import torch
from datasets import Dataset
from huggingface_hub import HfApi, hf_hub_download
from IPython.display import display, Markdown
from tqdm.auto import tqdm
from transformers import TrainerCallback, set_seed
from transformers.trainer_utils import get_last_checkpoint
from sentence_transformers import (
    MultiVectorEncoder,
    MultiVectorEncoderTrainer,
    MultiVectorEncoderTrainingArguments,
    SentenceTransformer,
)
from sentence_transformers.base.sampler import BatchSamplers
from sentence_transformers.multi_vector_encoder.evaluation import MultiVectorInformationRetrievalEvaluator
from sentence_transformers.multi_vector_encoder.losses import CachedMultiVectorMultipleNegativesRankingLoss
from sentence_transformers.util.similarity import maxsim, maxsim_pairwise

if not torch.cuda.is_available():
    raise RuntimeError("Keine CUDA-GPU gefunden. In Colab: Laufzeit → Laufzeittyp ändern → GPU.")

IN_COLAB = "google.colab" in sys.modules
WORK_ROOT = Path("/content") if IN_COLAB else Path.cwd()
GPU_NAME = torch.cuda.get_device_name(0)
CAPABILITY = torch.cuda.get_device_capability(0)
VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
RAM_GIB = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
DISK_FREE_GIB = shutil.disk_usage(WORK_ROOT).free / 2**30
print(f"GPU {GPU_NAME} (sm_{CAPABILITY[0]}{CAPABILITY[1]}, {VRAM_GIB:.1f} GiB) · RAM {RAM_GIB:.1f} GiB · "
      f"frei auf {WORK_ROOT}: {DISK_FREE_GIB:.0f} GiB · Colab: {'ja' if IN_COLAB else 'nein'}")
print(f"PyTorch {torch.__version__} · Sentence Transformers {version('sentence-transformers')} · "
      f"Transformers {version('transformers')}")


def colab_secret(name):
    """Umgebungsvariable, sonst Colab-Secret (Schlüssel-Symbol links). None, wenn nicht gesetzt oder nicht freigegeben."""
    value = os.environ.get(name)
    if value or not IN_COLAB:
        return value or None
    try:
        from google.colab import userdata
        return userdata.get(name) or None
    except Exception as error:   # SecretNotFoundError, NotebookAccessError, TimeoutException
        print(f"Colab-Secret {name} nicht verfügbar ({type(error).__name__}).")
        return None


def mount_drive():
    """Google Drive einhängen (nur Colab). Liefert MyDrive oder None."""
    if not IN_COLAB:
        return None
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    return Path("/content/drive/MyDrive")


TIMINGS = {}


@contextmanager
def timed(phase):
    start = time.time()
    try:
        yield
    finally:
        TIMINGS[phase] = TIMINGS.get(phase, 0.0) + (time.time() - start) / 60


# Auswertungsmodul des LLM-Judges (Prompt, Metriken, Cache, Budget) aus dem öffentlichen Datensatz-Repo.
JUDGE_DATASET = "johannhartmann/esci-llm-judged-top10"
JUDGE_REVISION = "483d780170a13641eb585acae2cba32b7cfee4f5"
HF_TOKEN = os.environ.get("HF_TOKEN") or None    # Lesen geht ohne Token; Schreiben holt es in Abschnitt 11


def judge_file(path):
    return hf_hub_download(JUDGE_DATASET, path, repo_type="dataset", revision=JUDGE_REVISION, token=HF_TOKEN)


JUDGE_CODE = Path(judge_file("code/esci_judge.py")).parent
sys.path.insert(0, str(JUDGE_CODE))
import esci_judge as ej
print("Hilfsmodul:", JUDGE_CODE / "esci_judge.py")

## 2 · Profil und Konfiguration

Das Profil bestimmt Trainingsumfang und Suchkatalog. `auto` wählt nach GPU-Speicher, RAM und freier Platte.

| Profil | Training | Suchkatalog der Evaluation | Test-Queries | Platte | Auto-Wahl |
|---|---|---|---|---|---|
| `t4` | 3.000 Queries, ≤2 Exact-Produkte je Query (~5.600 Tripel) | 100.000 Produkte | 1.000 | ~25 GB | Standard |
| `l4` | 6.000 Queries, ≤2 je Query | 150.000 Produkte | 1.000 | ~35 GB | ≥22 GB GPU, ≥24 GB RAM |
| `a100` | alle ~20k Queries, ≤4 je Query (~68k Tripel, Rezept des veröffentlichten Modells) | alle 482.105 Produkte | 2.000 | ~75 GB | ≥38 GB GPU, ≥45 GB RAM, ≥100 GB Platte |
| `full` | wie `a100` | alle 482.105 Produkte | alle 8.955 | ~75 GB | nur manuell |

**Suchkatalog:** alle ESCI-beurteilten Produkte der verwendeten Queries (Training, Dev, Test) plus zufällige
weitere Produkte bis zur Profilgröße. Gesucht wird exakt über diesen Katalog. Ein kleinerer Katalog ist leichter:
Zahlen aus `t4`/`l4` sind untereinander und mit dem veröffentlichten Modell *im selben Katalog* vergleichbar, nicht
1:1 mit der Tabelle über den vollen Katalog. Die ersten 1.000 Test-Queries sind genau die Stichprobe des
LLM-Judge-Datensatzes (Abschnitt 9b).

**Mixed Precision:** BF16 ab Ampere (A100, L4), FP16 auf T4. **Fortsetzen:** Mit `USE_GOOGLE_DRIVE=True` liegen
Trainingstripel, Checkpoints (~7 GB), Modell und Ergebnisse auf Drive. Nach einem Abbruch mit gleichem Profil erneut
*Alle ausführen*: Das Training setzt am letzten Checkpoint fort, ein fertiges Modell wird nicht neu trainiert.

In [ ]:
# @title Konfiguration
PROFILE = "auto"             # @param ["auto", "t4", "l4", "a100", "full"]
PRECISION = "auto"           # @param ["auto", "bf16", "fp16"]
USE_GOOGLE_DRIVE = False     # @param {type:"boolean"}
RESUME = True                # @param {type:"boolean"}
RUN_NAME = ""                # @param {type:"string"}
COMPARE_PUBLISHED = True     # @param {type:"boolean"}
EXTRA_BASELINES = False      # @param {type:"boolean"}   zusätzlich mDenseOn (Dense) und LightOns mLateOn

PROFILES = {
    "t4": dict(train_queries=3000, positives_per_query=2, dev_queries=300, dev_distractors=5_000,
               eval_queries=1000, catalog_size=100_000, mini_batch_size=16, encode_batch_size=64),
    "l4": dict(train_queries=6000, positives_per_query=2, dev_queries=500, dev_distractors=10_000,
               eval_queries=1000, catalog_size=150_000, mini_batch_size=16, encode_batch_size=128),
    "a100": dict(train_queries=None, positives_per_query=4, dev_queries=1000, dev_distractors=30_000,
                 eval_queries=2000, catalog_size=None, mini_batch_size=32, encode_batch_size=128),
    "full": dict(train_queries=None, positives_per_query=4, dev_queries=1000, dev_distractors=30_000,
                 eval_queries=None, catalog_size=None, mini_batch_size=32, encode_batch_size=128),
}
DISK_NEED_GIB = {"t4": 25, "l4": 35, "a100": 75, "full": 75}


def auto_profile():
    if VRAM_GIB >= 38 and RAM_GIB >= 45 and DISK_FREE_GIB >= 100:
        return "a100"
    if VRAM_GIB >= 22 and RAM_GIB >= 24 and DISK_FREE_GIB >= 40:
        return "l4"
    return "t4"


if PROFILE == "auto":
    PROFILE = auto_profile()
P = PROFILES[PROFILE]
if DISK_FREE_GIB < DISK_NEED_GIB[PROFILE]:
    print(f"Achtung: Profil {PROFILE} braucht ~{DISK_NEED_GIB[PROFILE]} GB, frei sind {DISK_FREE_GIB:.0f} GB.")

if PRECISION == "auto":
    BF16 = CAPABILITY[0] >= 8        # T4/V100 (sm_75/sm_70) haben kein natives BF16
elif PRECISION == "bf16" and CAPABILITY[0] < 8:
    raise ValueError(f"{GPU_NAME} rechnet BF16 nur emuliert – PRECISION='fp16' oder 'auto' verwenden.")
else:
    BF16 = PRECISION == "bf16"
AMP_DTYPE = torch.bfloat16 if BF16 else torch.float16

BASE_MODEL = "lightonai/mLateOn-unsupervised"
PUBLISHED_MODEL = "johannhartmann/mlateon-esci-product-search" if COMPARE_PUBLISHED else ""
PUBLISHED_REVISION = "23afaa106fc4eda0e86bbc46d9518d9675c98a9e"
REFERENCE_MODEL = "lightonai/mLateOn" if EXTRA_BASELINES else ""
DENSE_MODEL = "lightonai/mDenseOn" if EXTRA_BASELINES else ""

EPOCHS = 1.0
LEARNING_RATE = 1e-5
LOSS_SCALE = 30.0                       # 1/Temperatur im InfoNCE-Loss (Abschnitt 6)
BATCH_SIZE = 128
MINI_BATCH_SIZE = P["mini_batch_size"]  # GradCache-Paket: bestimmt den Speicherbedarf (16 → ~8 GiB)
QUERY_TOKENS = 32
DOCUMENT_TOKENS = 256
ENCODE_BATCH_SIZE = P["encode_batch_size"]
MINING_TOP_K = 500
MINING_RELATIVE_MARGIN = 0.01
SEED = 42
NEGATIVES = "labeled+mined"
ESCI_REVISION = "7916cdf6ab75a462e77f20ab40428a10923998d5"
LOCALE = "us"

LOCAL_ROOT = WORK_ROOT / "colbert_training"          # Rohdaten und Indizes: lokal, reproduzierbar
PERSIST_ROOT = LOCAL_ROOT
if USE_GOOGLE_DRIVE:
    drive_root = mount_drive()
    if drive_root:
        PERSIST_ROOT = drive_root / "colbert_training"
    else:
        print("USE_GOOGLE_DRIVE wirkt nur in Colab – Lauf bleibt lokal.")

RUN_NAME = RUN_NAME or f"mlateon-esci-colab-{PROFILE}"
DATA_DIR = LOCAL_ROOT / "data"
INDEX_DIR = LOCAL_ROOT / "index"
RUN_DIR = PERSIST_ROOT / "runs" / RUN_NAME            # Tripel, Checkpoints, Modell, Ergebnisse
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
MODEL_DIR = RUN_DIR / "model"
for directory in (DATA_DIR, INDEX_DIR, RUN_DIR):
    directory.mkdir(parents=True, exist_ok=True)
ALREADY_TRAINED = RESUME and (MODEL_DIR / "training_recipe.json").exists()
set_seed(SEED)
print(f"Profil {PROFILE} · {'BF16' if BF16 else 'FP16'} · Lauf {RUN_DIR}"
      + (" · fertiges Modell gefunden, Training wird übersprungen" if ALREADY_TRAINED else ""))

## 3 · Amazon ESCI laden

Englischer Teil (`us`) der Ranking-Aufgabe (`small_version=1`) mit offiziellem Split, Apache-2.0.
Die Parquet-Dateien (1,2 GB) liegen in Git LFS und kommen von `media.githubusercontent.com`.

| Label | Bedeutung | Training | nDCG-Gain |
|---|---|---|---|
| E | Exact | Positiv | 1.0 |
| S | Substitute | weder positiv noch negativ | 0.1 |
| C | Complement | Negativ | 0.01 |
| I | Irrelevant | Negativ | 0.0 |

In [ ]:
ESCI_BASE = f"https://media.githubusercontent.com/media/amazon-science/esci-data/{ESCI_REVISION}/shopping_queries_dataset"
ESCI_FILES = {
    "shopping_queries_dataset_examples.parquet": 51_286_808,
    "shopping_queries_dataset_products.parquet": 1_108_857_465,
}


def download_parquet(filename, expected_bytes):
    path = DATA_DIR / filename
    if path.exists() and path.stat().st_size == expected_bytes:
        return path
    partial = path.with_suffix(".parquet.part")
    for attempt in range(3):
        try:
            with requests.get(f"{ESCI_BASE}/{filename}", stream=True, timeout=(20, 120)) as response:
                response.raise_for_status()
                with partial.open("wb") as out, tqdm(total=expected_bytes, unit="B", unit_scale=True,
                                                     desc=filename) as bar:
                    for chunk in response.iter_content(chunk_size=2**20):
                        out.write(chunk)
                        bar.update(len(chunk))
            break
        except requests.RequestException as error:
            if attempt == 2:
                raise
            print(f"Download abgebrochen ({error}), neuer Versuch …")
            time.sleep(10)
    with partial.open("rb") as downloaded:
        header = downloaded.read(4)
    if partial.stat().st_size != expected_bytes or header != b"PAR1":
        raise RuntimeError(f"Download unvollständig oder keine Parquet-Datei: {partial}")
    partial.replace(path)
    return path


with timed("ESCI laden"):
    examples_path = download_parquet(*list(ESCI_FILES.items())[0])
    products_path = download_parquet(*list(ESCI_FILES.items())[1])
print("ESCI bereit:", examples_path.parent)

### Queries, Splits und Katalog

- **Test:** Queries aus dem offiziellen Test-Split mit mindestens einem Exact-Produkt, nie trainiert. Zuerst die
  1.000 Queries der LLM-Judge-Stichprobe, danach weitere in fester Zufallsreihenfolge.
- **Dev:** zurückgehaltene Trainings-Queries; der Trainer misst darauf und wählt den besten Checkpoint.
- **Training:** übrige Trainings-Queries. Queries, die normalisiert auch im Test vorkommen, fliegen raus;
  widersprüchliche Urteile zum selben Paar werden verworfen.

In [ ]:
def normalize_query(values):
    return (pd.Series(values, dtype="string").str.normalize("NFKC").str.casefold()
            .str.replace(r"\s+", " ", regex=True).str.strip())


GAINS = {"E": 1.0, "S": 0.1, "C": 0.01, "I": 0.0}
RELEVANT_LABELS = {"E"}              # zählt für Recall/MRR und als Trainingspositiv
KNOWN_RELEVANT_LABELS = {"E", "S"}   # niemals als Negativ verwenden
NEGATIVE_LABELS = {"C", "I"}         # annotierte Negative

with timed("ESCI aufbereiten"):
    examples = pd.read_parquet(
        examples_path,
        columns=["query", "product_id", "product_locale", "esci_label", "small_version", "split"],
        filters=[("product_locale", "==", LOCALE), ("small_version", "==", 1)],
    )
    examples["query_key"] = normalize_query(examples["query"]).to_numpy()
    examples = examples[examples["query_key"].ne("")]
    n_labels = examples.groupby(["query_key", "product_id"])["esci_label"].transform("nunique")
    examples = examples[n_labels.eq(1)].drop_duplicates(["query_key", "product_id"]).reset_index(drop=True)

    queries = examples.drop_duplicates("query_key").set_index("query_key")["query"]
    keys_with_exact = set(examples.loc[examples["esci_label"].eq("E"), "query_key"])
    test_side = set(examples.loc[examples["split"].eq("test"), "query_key"])
    rng = random.Random(SEED)
    test_keys = sorted(test_side & keys_with_exact)
    train_pool = sorted((set(examples["query_key"]) - test_side) & keys_with_exact)
    rng.shuffle(test_keys)
    rng.shuffle(train_pool)

    # Die Judge-Stichprobe zuerst: So decken die LLM-Urteile vom Hub die Test-Queries jedes Profils ab.
    judge_queries = pd.read_parquet(judge_file("data/queries.parquet"))
    judge_sample = [k for k in judge_queries["query_key"] if k in set(test_keys)]
    in_sample = set(judge_sample)
    test_keys = judge_sample + [k for k in test_keys if k not in in_sample]
    eval_keys = test_keys[:P["eval_queries"]] if P["eval_queries"] else test_keys
    dev_keys = train_pool[:P["dev_queries"]]
    train_keys = train_pool[P["dev_queries"]:]
    if P["train_queries"]:
        train_keys = train_keys[:P["train_queries"]]

    used = examples[examples["query_key"].isin(set(eval_keys) | set(dev_keys) | set(train_keys))]
    judged = {key: dict(zip(group["product_id"], group["esci_label"])) for key, group in used.groupby("query_key")}

    all_product_ids = sorted(examples["product_id"].unique())
    if P["catalog_size"]:
        required = set(used["product_id"])
        rest = sorted(set(all_product_ids) - required)
        rng.shuffle(rest)
        catalog_ids = sorted(required | set(rest[:max(0, P["catalog_size"] - len(required))]))
    else:
        catalog_ids = all_product_ids

print(f"Queries – Training: {len(train_keys):,} · Dev: {len(dev_keys):,} · Test: {len(eval_keys):,} "
      f"(davon Judge-Stichprobe {len(in_sample & set(eval_keys)):,})")
print(f"Urteile gesamt: {len(examples):,} · Katalog: {len(catalog_ids):,} von {len(all_product_ids):,} Produkten")
del examples, used
_ = gc.collect()

### Produkttexte

Die 1,1-GB-Produktdatei wird batchweise gelesen; nur die Katalogprodukte bleiben im Speicher. Modellinput: Titel,
Marke, Farbe, Stichpunkte, Beschreibung – strukturierte Felder zuerst, weil bei `DOCUMENT_TOKENS` abgeschnitten wird.

In [ ]:
def clean_text(value):
    if not isinstance(value, str):
        return ""
    value = re.sub(r"<[^>]+>", " ", value)
    return re.sub(r"\s+", " ", html.unescape(value)).strip()


def product_text(row):
    fields = [("Product", "product_title"), ("Brand", "product_brand"), ("Color", "product_color"),
              ("Features", "product_bullet_point"), ("Description", "product_description")]
    return "\n".join(f"{label}: {text}" for label, field in fields if (text := clean_text(row[field])))


def title_key(titles):
    return pd.Series(titles, dtype="string").fillna("").str.casefold().str.replace(r"\W+", " ", regex=True).str.strip()


with timed("ESCI aufbereiten"):
    wanted = set(catalog_ids)
    parts = []
    product_file = pq.ParquetFile(products_path)
    columns = ["product_id", "product_locale", "product_title", "product_brand", "product_color",
               "product_bullet_point", "product_description"]
    for batch in tqdm(product_file.iter_batches(batch_size=50_000, columns=columns), desc="Produkttexte lesen"):
        frame = batch.to_pandas()
        frame = frame[frame["product_locale"].eq(LOCALE) & frame["product_id"].isin(wanted)]
        if len(frame):
            frame = frame.drop_duplicates("product_id")
            parts.append(pd.DataFrame({
                "product_id": frame["product_id"].to_numpy(),
                "title": frame["product_title"].map(clean_text).to_numpy(),
                "text": frame.apply(product_text, axis=1).to_numpy(),
            }))
    catalog = pd.concat(parts, ignore_index=True).drop_duplicates("product_id")
    catalog = catalog[catalog["text"].ne("")].set_index("product_id").sort_index()
    catalog["title_key"] = title_key(catalog["title"]).to_numpy()
    del parts, frame, product_file
    _ = gc.collect()
print(f"Katalog: {len(catalog):,} Produkte mit Text ({len(wanted) - len(catalog):,} ohne Text verworfen)")

## 4 · Basismodell laden

[`mLateOn-unsupervised`](https://huggingface.co/lightonai/mLateOn-unsupervised) (mmBERT-base, 128-dim
Token-Vektoren, MaxSim) ist nur kontrastiv vortrainiert und als Startpunkt für eigenes Fine-Tuning gedacht.
`model` hält FP32-Gewichte für das Training (Mixed Precision rechnet in BF16/FP16), **alle Parameter trainierbar**;
`base_encoder` ist eine unveränderte Kopie in BF16/FP16 für Mining, Evaluation und den Vorher/Nachher-Vergleich.

In [ ]:
BASE_REVISION = HfApi().model_info(BASE_MODEL).sha


def load_encoder(name_or_path, dtype, revision=None):
    encoder = MultiVectorEncoder(
        name_or_path, revision=revision, device="cuda",
        model_kwargs={"dtype": dtype, "attn_implementation": "sdpa"},
    )
    encoder[0].query_length = QUERY_TOKENS
    encoder[0].document_length = DOCUMENT_TOKENS
    return encoder


base_encoder = load_encoder(BASE_MODEL, AMP_DTYPE, BASE_REVISION).eval()
if not ALREADY_TRAINED:
    model = load_encoder(BASE_MODEL, torch.float32, BASE_REVISION)
    for parameter in model.parameters():
        parameter.requires_grad_(True)
    print(f"{sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.0f} Mio. trainierbare Parameter")
print(f"{BASE_MODEL}@{BASE_REVISION[:10]} · Marker: {base_encoder.prompts}")

## 5 · Katalog-Index und Hard Negatives

**Index:** Jedes Produkt wird zu einer Matrix aus Token-Vektoren (im Mittel ~165 × 128, FP16, ~40 KB je Produkt).
Gesucht wird **exakt**: MaxSim jeder Query gegen jedes Katalogprodukt, blockweise auf der GPU. Keine Approximation.

**Hard Negatives:** Die Suche mit dem Basismodell liefert je Trainings-Query die 500 ähnlichsten Produkte. Als
Negativ taugt nur, was nicht als E/S beurteilt ist, keinen identischen Titel wie ein E/S-Produkt der Query hat und
mindestens 1 % unter dem Score des besten relevanten Produkts liegt (Positiv-relative Marge nach
[NV-Retriever](https://huggingface.co/papers/2407.15831)). Je Tripel zwei Negative: ein annotiertes C/I-Produkt
(sonst ein zweites gemintes) plus ein gemintes.

In [ ]:
class TokenIndex:
    """Token-Vektoren eines Katalogs als FP16-Datei + Offsets; exakte MaxSim-Suche blockweise auf der GPU."""

    def __init__(self, directory):
        self.directory = Path(directory)
        meta = json.loads((self.directory / "meta.json").read_text())
        self.doc_ids = meta["doc_ids"]
        self.position = {doc_id: i for i, doc_id in enumerate(self.doc_ids)}
        self.lengths = np.load(self.directory / "lengths.npy").astype(np.int64)
        self.offsets = np.concatenate([[0], np.cumsum(self.lengths)])
        self.vectors = np.memmap(self.directory / "vectors.f16", dtype=np.float16, mode="r",
                                 shape=(int(self.offsets[-1]), meta["dim"]))

    @property
    def nbytes(self):
        return self.vectors.nbytes

    @classmethod
    def build(cls, encoder, doc_ids, texts, directory, fingerprint, block=16_384):
        directory = Path(directory)
        meta_path = directory / "meta.json"
        if meta_path.exists():
            meta = json.loads(meta_path.read_text())
            if meta["fingerprint"] == fingerprint and meta["doc_ids"] == list(doc_ids):
                print(f"Index wiederverwendet: {directory}")
                return cls(directory)
        directory.mkdir(parents=True, exist_ok=True)
        meta_path.unlink(missing_ok=True)
        lengths, dim = [], None
        encoder.eval()
        with open(directory / "vectors.f16", "wb") as out, torch.inference_mode():
            for start in tqdm(range(0, len(texts), block), desc=f"Katalog einbetten ({directory.name})"):
                with torch.autocast("cuda", dtype=AMP_DTYPE):
                    embeddings = encoder.encode_document(list(texts[start:start + block]),
                                                         batch_size=ENCODE_BATCH_SIZE, show_progress_bar=False)
                lengths.extend(e.shape[0] for e in embeddings)
                dim = embeddings[0].shape[1]
                out.write(torch.cat(embeddings).to("cpu", torch.float16).numpy().tobytes())
        np.save(directory / "lengths.npy", np.asarray(lengths, dtype=np.int32))
        meta_path.write_text(json.dumps({"fingerprint": fingerprint, "dim": dim, "doc_ids": list(doc_ids)}))
        return cls(directory)

    def document_vectors(self, positions):
        return [torch.from_numpy(np.array(self.vectors[self.offsets[p]:self.offsets[p + 1]])) for p in positions]

    @torch.inference_mode()
    def search(self, query_embeddings, k=100, block_tokens=4_000_000, query_batch=512,
               chunk_elements=500_000_000, progress=True):
        """Top-k (Score, Katalogposition) je Query über den gesamten Katalog, exakt per MaxSim."""
        device = torch.device("cuda")
        queries = [q.to(device, torch.float16) for q in query_embeddings]
        order = np.argsort([len(q) for q in queries], kind="stable")   # ähnliche Längen → wenig Padding
        batches = []
        for i in range(0, len(order), query_batch):
            rows = order[i:i + query_batch]
            batch = [queries[r] for r in rows]
            mask = torch.nn.utils.rnn.pad_sequence(
                [torch.ones(len(q), dtype=torch.bool, device=device) for q in batch], batch_first=True)
            batches.append((torch.as_tensor(rows, device=device),
                            torch.nn.utils.rnn.pad_sequence(batch, batch_first=True), mask))
        best_scores = torch.full((len(queries), k), float("-inf"), device=device)
        best_positions = torch.full((len(queries), k), -1, dtype=torch.long, device=device)
        n_docs, start = len(self.lengths), 0
        with tqdm(total=n_docs, desc="MaxSim-Suche", disable=not progress, unit="doc") as bar:
            while start < n_docs:
                end = int(np.searchsorted(self.offsets, self.offsets[start] + block_tokens, side="right")) - 1
                end = min(max(end, start + 1), n_docs)
                tokens = torch.from_numpy(np.array(self.vectors[self.offsets[start]:self.offsets[end]])).to(device)
                documents = list(torch.split(tokens, self.lengths[start:end].tolist()))
                positions = torch.arange(start, end, device=device)
                for rows, q, q_mask in batches:
                    scores = maxsim(q, documents, a_mask=q_mask, chunk_elements=chunk_elements)
                    merged = torch.cat([best_scores[rows], scores], dim=1)
                    merged_pos = torch.cat([best_positions[rows], positions.expand(len(rows), -1)], dim=1)
                    top = merged.topk(k, dim=1)
                    best_scores[rows] = top.values
                    best_positions[rows] = merged_pos.gather(1, top.indices)
                bar.update(end - start)
                start = end
        return best_scores.cpu(), best_positions.cpu()


@torch.inference_mode()
def encode_queries(encoder, texts):
    encoder.eval()
    with torch.autocast("cuda", dtype=AMP_DTYPE):
        return encoder.encode_query(list(texts), batch_size=ENCODE_BATCH_SIZE, show_progress_bar=False)


catalog_fingerprint = hashlib.sha1("\n".join(catalog.index).encode()).hexdigest()[:10]
with timed("Index Basismodell"):
    base_index = TokenIndex.build(
        base_encoder, catalog.index, catalog["text"].to_numpy(),
        INDEX_DIR / f"base-{catalog_fingerprint}",
        fingerprint=f"{BASE_MODEL}@{BASE_REVISION}|q{QUERY_TOKENS}|d{DOCUMENT_TOKENS}|{AMP_DTYPE}",
    )
print(f"Basis-Index: {len(base_index.doc_ids):,} Produkte, {base_index.vectors.shape[0]:,} Token-Vektoren, "
      f"{base_index.nbytes / 2**30:.1f} GiB")

In [ ]:
DISPLAY_BLOCKLIST = re.compile(
    r"\b(sex\w*|dildo\w*|vibrat\w*|porn\w*|erotic\w*|fetish\w*|bdsm|bondage|lingerie|condom\w*|lube|lubricant\w*|"
    r"masturbat\w*|anal|penis\w*|vagina\w*|butt ?plug\w*|nipple\w*|strap-?on\w*|orgasm\w*|kinky|nude|naked|"
    r"stripper\w*|thong\w*|g-?string\w*|adult toys?|pleasure|intimate|escort|thc|cannabis|marijuana|weed|stoner)\b",
    re.IGNORECASE)


def presentable(*texts):
    """Nur für die Anzeige: keine Beispiele mit Erotik-/Adult-Bezug."""
    return not any(DISPLAY_BLOCKLIST.search(str(t)) for t in texts)


rng = random.Random(SEED)
pair_rows = []
for key in train_keys:
    positives = sorted(d for d, label in judged[key].items() if label in RELEVANT_LABELS and d in base_index.position)
    rng.shuffle(positives)
    for positive_id in positives[:P["positives_per_query"] or None]:
        pair_rows.append((key, positive_id))
pairs = pd.DataFrame(pair_rows, columns=["query_key", "positive_id"])


def mine_hard_negatives(encoder, index, pairs, num_negatives):
    """Hard Negatives per exakter MaxSim-Suche des Basismodells über den Katalog."""
    keys = list(dict.fromkeys(pairs["query_key"]))
    key_row = {k: i for i, k in enumerate(keys)}
    query_embeddings = encode_queries(encoder, queries.loc[keys])
    cache = index.directory / f"mining-{hashlib.sha1(json.dumps([keys, MINING_TOP_K]).encode()).hexdigest()[:12]}.pt"
    if cache.exists():
        cand_scores, cand_positions = torch.load(cache)
    else:
        cand_scores, cand_positions = index.search(query_embeddings, k=MINING_TOP_K)
        torch.save((cand_scores, cand_positions), cache)

    positive_scores = []
    for start in range(0, len(pairs), 4096):
        chunk = pairs.iloc[start:start + 4096]
        q = [query_embeddings[key_row[k]].to("cuda", torch.float16) for k in chunk["query_key"]]
        d = [v.to("cuda") for v in index.document_vectors([index.position[p] for p in chunk["positive_id"]])]
        positive_scores.append(maxsim_pairwise(q, d).float().cpu())
    positive_scores = torch.cat(positive_scores).numpy()
    best_positive = pd.Series(positive_scores).groupby(pairs["query_key"].to_numpy()).max()

    title_keys = catalog["title_key"].reindex(index.doc_ids).to_numpy()
    mined, rejected_relevant, rejected_margin = [], 0, 0
    for key in pairs["query_key"]:
        pos_score = best_positive[key]
        labels = judged[key]
        blocked_titles = {catalog.at[d, "title_key"] for d, l in labels.items()
                          if l in KNOWN_RELEVANT_LABELS and d in catalog.index} - {""}
        limit = pos_score - abs(pos_score) * MINING_RELATIVE_MARGIN
        chosen, row = [], key_row[key]
        for score, position in zip(cand_scores[row].tolist(), cand_positions[row].tolist()):
            if labels.get(index.doc_ids[position]) in KNOWN_RELEVANT_LABELS or title_keys[position] in blocked_titles:
                rejected_relevant += 1
            elif score > limit:
                rejected_margin += 1
            else:
                chosen.append(index.doc_ids[position])
                if len(chosen) == num_negatives:
                    break
        mined.append(chosen)
    print(f"Mining: {rejected_relevant:,} Kandidaten als bekannt relevant/Duplikat verworfen, "
          f"{rejected_margin:,} wegen zu geringem Abstand zum Positiv.")
    return mined


def labeled_negatives(pairs, per_pair=2):
    """Annotierte Negative (C/I) – reihum je Query, damit mehrere Positive verschiedene Negative sehen."""
    pools, result = {}, []
    for key in pairs["query_key"]:
        if key not in pools:
            pool = sorted(d for d, l in judged[key].items() if l in NEGATIVE_LABELS and d in catalog.index)
            random.Random(f"{SEED}-{key}").shuffle(pool)
            pools[key] = [pool, 0]
        pool, i = pools[key]
        result.append([pool[(i + j) % len(pool)] for j in range(min(per_pair, len(pool)))])
        pools[key][1] += per_pair
    return result


train_file = RUN_DIR / "train_triplets.parquet"
recipe_key = json.dumps(["v2", NEGATIVES, MINING_TOP_K, MINING_RELATIVE_MARGIN, BASE_REVISION, catalog_fingerprint,
                         len(pairs), SEED])
with timed("Hard-Negative-Mining"):
    if train_file.exists() and (RUN_DIR / "train_triplets.key").read_text() == recipe_key:
        triplets = pd.read_parquet(train_file)
        print("Trainingstripel aus vorherigem Lauf wiederverwendet (identische Einstellungen).")
    else:
        triplets = pairs.copy()
        labeled = labeled_negatives(triplets)
        mined = mine_hard_negatives(base_encoder, base_index, triplets, num_negatives=2)
        choices = [list(dict.fromkeys(l[:1] + m[:1] + l[1:] + m[1:]))[:2] for l, m in zip(labeled, mined)]
        triplets["negative_1_id"] = [c[0] if len(c) == 2 else None for c in choices]
        triplets["negative_2_id"] = [c[1] if len(c) == 2 else None for c in choices]
        triplets["negative_sources"] = [
            "+".join("annotiert" if d in l else "gemined" for d in c) for c, l in zip(choices, labeled)]
        before = len(triplets)
        triplets = triplets.dropna(subset=["negative_1_id", "negative_2_id"]).reset_index(drop=True)
        print(f"{before - len(triplets):,} von {before:,} Paaren ohne geeignetes Negativ verworfen.")
        print("Herkunft der Negative:", triplets["negative_sources"].value_counts().to_dict())
        triplets["query"] = queries.loc[triplets["query_key"]].to_numpy()
        triplets["positive"] = catalog.loc[triplets["positive_id"], "text"].to_numpy()
        for column in ["negative_1_id", "negative_2_id"]:
            triplets[column.removesuffix("_id")] = catalog.loc[triplets[column], "text"].to_numpy()
        triplets = triplets.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
        triplets.to_parquet(train_file)
        (RUN_DIR / "train_triplets.key").write_text(recipe_key)

negative_id_columns = ["negative_1_id", "negative_2_id"]
text_columns = ["query", "positive", "negative_1", "negative_2"]
train_dataset = Dataset.from_pandas(triplets[text_columns], preserve_index=False)
print(f"Trainingsdaten: {len(train_dataset):,} Tripel aus {triplets['query_key'].nunique():,} Queries")

preview = []
for row in triplets.itertuples(index=False):
    titles = [catalog.at[row.positive_id, "title"]] + [catalog.at[getattr(row, c), "title"] for c in negative_id_columns]
    if presentable(row.query, *titles):
        preview.append({"Query": row.query, "Positiv": titles[0], "Negativ 1": titles[1], "Negativ 2": titles[2]})
    if len(preview) == 5:
        break
display(pd.DataFrame(preview))

## 6 · Training konfigurieren

- **Loss:** `CachedMultiVectorMultipleNegativesRankingLoss` – InfoNCE über MaxSim-Scores. Jede Query muss ihr
  Positiv gegen die eigenen zwei Hard Negatives **und** alle Produkte der anderen 127 Queries im Batch gewinnen.
  GradCache rechnet den 128er-Batch in Paketen von `MINI_BATCH_SIZE`; der Speicherbedarf hängt vom Paket ab.
- **Temperatur `LOSS_SCALE=30`:** Die MaxSim-Scores dieses Checkpoints liegen eng beieinander; mit dem Default 1
  ist die Softmax fast flach und das Modell wird schlechter (Dev-nDCG@10 0,548 → 0,470; Skala 30 → 0,580).
- **Sampler `NO_DUPLICATES`:** keine Query und kein Produkt zweimal im Batch.
- **Dev-Evaluation:** Dev-Queries gegen ihre beurteilten Produkte plus Distraktoren, vor dem Training und
  alle `EVAL_STEPS`. Jeder Evaluationspunkt ist ein Checkpoint; am Ende lädt der Trainer den besten (Dev-nDCG@10).

In [ ]:
steps_per_epoch = math.ceil(len(train_dataset) / BATCH_SIZE)
total_steps = math.ceil(steps_per_epoch * EPOCHS)
EVAL_STEPS = max(25, math.ceil(total_steps / 8))

if ALREADY_TRAINED:
    print("Fertiges Modell unter", MODEL_DIR, "– Training wird übersprungen (RESUME=False trainiert neu).")
else:
    rng = random.Random(SEED + 1)
    dev_corpus_ids = {d for k in dev_keys for d in judged[k] if d in catalog.index}
    distractor_pool = sorted(set(catalog.index) - dev_corpus_ids)
    dev_corpus_ids |= set(rng.sample(distractor_pool, min(P["dev_distractors"], len(distractor_pool))))
    dev_relevant = {k: {d for d, l in judged[k].items() if l in RELEVANT_LABELS and d in dev_corpus_ids}
                    for k in dev_keys}
    dev_relevant = {k: v for k, v in dev_relevant.items() if v}

    dev_evaluator = MultiVectorInformationRetrievalEvaluator(
        queries={k: queries[k] for k in dev_relevant},
        corpus={d: catalog.at[d, "text"] for d in sorted(dev_corpus_ids)},
        relevant_docs=dev_relevant,
        name="dev",
        batch_size=ENCODE_BATCH_SIZE,
        mrr_at_k=[10], ndcg_at_k=[10], accuracy_at_k=[1, 10], precision_recall_at_k=[10, 100], map_at_k=[100],
    )
    with timed("Dev-Evaluation vorher"), torch.autocast("cuda", dtype=AMP_DTYPE):
        dev_before = dev_evaluator(model)
    DEV_METRIC = f"eval_{dev_evaluator.primary_metric}"   # erst nach dem ersten Aufruf gesetzt
    (RUN_DIR / "dev_before.json").write_text(json.dumps(dev_before, indent=2))
    print(f"Dev vor dem Training ({len(dev_relevant):,} Queries, {len(dev_corpus_ids):,} Produkte): "
          + ", ".join(f"{k.split('_', 2)[-1]}={v:.4f}" for k, v in dev_before.items()
                      if "ndcg" in k or "recall@100" in k))

    loss = CachedMultiVectorMultipleNegativesRankingLoss(model, scale=LOSS_SCALE, mini_batch_size=MINI_BATCH_SIZE)
    args = MultiVectorEncoderTrainingArguments(
        output_dir=str(CHECKPOINT_DIR),
        run_name=RUN_NAME,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        warmup_steps=0.05,
        lr_scheduler_type="linear",
        weight_decay=0.01,
        max_grad_norm=1.0,
        bf16=BF16,
        fp16=not BF16,
        prompts={column: base_encoder.prompts["query" if column == "query" else "document"] for column in text_columns},
        batch_sampler=BatchSamplers.NO_DUPLICATES,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        save_strategy="steps",
        save_steps=EVAL_STEPS,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model=DEV_METRIC,
        greater_is_better=True,
        logging_steps=10,
        logging_first_step=True,
        report_to="none",
        seed=SEED,
        data_seed=SEED,
    )

    class EvaluateAtEnd(TrainerCallback):
        """Auch der letzte Schritt wird evaluiert und gespeichert, damit er für den besten Checkpoint zählt."""

        def on_step_end(self, args, state, control, **kwargs):
            if state.global_step >= state.max_steps:
                control.should_evaluate = control.should_save = True

    trainer = MultiVectorEncoderTrainer(
        model=model, args=args, train_dataset=train_dataset, loss=loss, evaluator=dev_evaluator,
        callbacks=[EvaluateAtEnd()],
    )
    print(f"{len(train_dataset):,} Tripel · Batch {BATCH_SIZE} (GradCache-Paket {MINI_BATCH_SIZE}) · "
          f"{total_steps:,} Schritte · Dev-Evaluation/Checkpoint alle {EVAL_STEPS} Schritte · "
          f"{'BF16' if BF16 else 'FP16'}")

## 7 · Trainieren

Forward, Backward und Optimierung aller Gewichte auf der GPU. Bei CUDA-Out-of-Memory `MINI_BATCH_SIZE`
im Profil halbieren – die Trainingsdynamik bleibt gleich, nur langsamer. Nach einem Abbruch setzt der nächste Lauf
am letzten Checkpoint fort.

In [ ]:
PEAK_BEFORE_TRAINING_GIB = torch.cuda.max_memory_allocated() / 2**30
if not ALREADY_TRAINED:
    last_checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR)) if RESUME and CHECKPOINT_DIR.exists() else None
    if last_checkpoint:
        print("Setze fort ab", last_checkpoint)
    model.train()
    torch.cuda.reset_peak_memory_stats()
    train_start = time.time()
    with timed("Training"):
        train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
    train_minutes = (time.time() - train_start) / 60
    training_peak_gib = torch.cuda.max_memory_allocated() / 2**30
    pd.DataFrame(trainer.state.log_history).to_json(RUN_DIR / "training_log.json", orient="records", indent=2)
    print(f"{trainer.state.global_step:,} Schritte in {train_minutes:.0f} min · "
          f"Spitzenbelegung (PyTorch): {training_peak_gib:.1f} GiB")
    print(f"Bester Checkpoint: {trainer.state.best_model_checkpoint} ({DEV_METRIC} = {trainer.state.best_metric:.4f})")

In [ ]:
import matplotlib.pyplot as plt

log = pd.read_json(RUN_DIR / "training_log.json")
dev_before = json.loads((RUN_DIR / "dev_before.json").read_text())
dev_metric = next(c for c in log.columns if c.startswith("eval_") and c.endswith("ndcg@10"))
fig, (ax_loss, ax_dev) = plt.subplots(1, 2, figsize=(12, 3.5))
curve = log.dropna(subset=["loss"])
ax_loss.plot(curve["step"], curve["loss"], lw=1)
ax_loss.set(xlabel="Schritt", ylabel="Trainings-Loss", title="Loss")
dev_curve = pd.concat([pd.DataFrame([{"step": 0, dev_metric: dev_before[dev_metric.removeprefix("eval_")]}]),
                       log.dropna(subset=[dev_metric])[["step", dev_metric]]])
ax_dev.plot(dev_curve["step"], dev_curve[dev_metric], marker="o")
ax_dev.set(xlabel="Schritt", ylabel="nDCG@10 (Dev)", title="Dev-Evaluation")
for ax in (ax_loss, ax_dev):
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8 · Speichern und neu laden

`save_pretrained` schreibt das vollständige Modell im Sentence-Transformers-Format (Gewichte, Projektionsköpfe,
Marker, Query-/Dokumentlänge, Tokenizer, Modellkarte), dazu `training_recipe.json` mit Revisionen, Parametern und
Laufmetriken. Ab hier arbeitet das Notebook **nur mit dem von der Platte geladenen Modell**.

In [ ]:
if not ALREADY_TRAINED:
    trainer.model.save_pretrained(str(MODEL_DIR))
    recipe = {
        "base_model": BASE_MODEL,
        "base_revision": BASE_REVISION,
        "data_source": "esci",
        "data": {"repository": "amazon-science/esci-data", "revision": ESCI_REVISION, "locale": LOCALE,
                 "positives": sorted(RELEVANT_LABELS), "labeled_negatives": sorted(NEGATIVE_LABELS)},
        "profile": PROFILE,
        "catalog_products": len(catalog),
        "negatives": NEGATIVES,
        "mining": {"top_k": MINING_TOP_K, "relative_margin": MINING_RELATIVE_MARGIN},
        "train_triplets": len(train_dataset),
        "train_queries": int(triplets["query_key"].nunique()),
        "dev_queries": len(dev_relevant),
        "query_tokens": QUERY_TOKENS,
        "document_tokens": DOCUMENT_TOKENS,
        "batch_size": BATCH_SIZE,
        "mini_batch_size": MINI_BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "loss_scale": LOSS_SCALE,
        "epochs": EPOCHS,
        "optimizer_steps": trainer.state.global_step,
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_dev_metric": {DEV_METRIC: trainer.state.best_metric},
        "dev_before_training": dev_before,
        "gpu": GPU_NAME,
        "mixed_precision": "bf16" if BF16 else "fp16",
        "train_minutes": round(train_minutes, 1),
        "training_peak_allocated_gib": round(training_peak_gib, 2),
        "seed": SEED,
        "packages": {name: version(name) for name in
                     ["torch", "sentence-transformers", "transformers", "datasets", "accelerate", "bm25s"]},
    }
    (MODEL_DIR / "training_recipe.json").write_text(json.dumps(recipe, indent=2), encoding="utf-8")
    del trainer, loss, model
    _ = gc.collect()
    torch.cuda.empty_cache()

recipe = json.loads((MODEL_DIR / "training_recipe.json").read_text(encoding="utf-8"))
trained_encoder = MultiVectorEncoder(str(MODEL_DIR), device="cuda", model_kwargs={"dtype": AMP_DTYPE}).eval()
print("Gespeichert und neu geladen:", MODEL_DIR)
print(f"Größe: {sum(p.stat().st_size for p in MODEL_DIR.rglob('*') if p.is_file()) / 2**30:.2f} GiB · "
      f"Query-/Dokumentlänge: {trained_encoder[0].query_length}/{trained_encoder[0].document_length} · "
      f"{recipe['optimizer_steps']} Schritte, bestes Dev-nDCG@10 {list(recipe['best_dev_metric'].values())[0]:.4f}")

## 9 · Retrieval-Evaluation auf dem Test-Split

Jedes Verfahren durchsucht **denselben Katalog** exakt nach jeder Test-Query; gewertet wird gegen die ESCI-Urteile:
**Recall@100** (Anteil der Exact-Produkte in den Top 100 – die Kennzahl aus der Kundengeschichte), Recall@10,
**nDCG@10** (Gains E=1, S=0,1, C=0,01, I=0; unbeurteilt zählt 0) und **MRR@10**. 95-%-Intervalle per Bootstrap
über die Queries.

Verglichen werden BM25 (`bm25s`, Stemming + Stopwords), das Basismodell, das veröffentlichte Vollmodell
(Profil `full`, alle Trainings-Queries) und das eigene Modell dieses Laufs.

**Zum Vergleich – voller Katalog** (Profil `full`, 8.955 Test-Queries, 482.105 Produkte, RTX A6000):

| Verfahren | Recall@10 | Recall@100 | nDCG@10 | MRR@10 |
|---|---|---|---|---|
| BM25 | 0,296 | 0,581 | 0,318 | 0,460 |
| mLateOn-unsupervised (Basis) | 0,362 | 0,669 | 0,392 | 0,549 |
| mlateon-esci-product-search | 0,406 | 0,723 | 0,443 | 0,606 |

In [ ]:
eval_query_texts = queries.loc[eval_keys].tolist()
eval_judged = [judged[k] for k in eval_keys]
TOP_K = 100


def per_query_metrics(ranked_ids):
    """ranked_ids: je Test-Query die Top-100-Produkt-IDs. Liefert einen DataFrame mit einer Zeile je Query."""
    rows = []
    for ranking, labels in zip(ranked_ids, eval_judged):
        relevant = {d for d, l in labels.items() if l in RELEVANT_LABELS}
        gains = [GAINS.get(labels.get(d), 0.0) for d in ranking[:10]]
        ideal = sorted((GAINS[l] for l in labels.values()), reverse=True)[:10]
        idcg = sum(g / math.log2(i + 2) for i, g in enumerate(ideal))
        first_hit = next((i for i, d in enumerate(ranking[:10]) if d in relevant), None)
        rows.append({
            "Recall@10": len(relevant & set(ranking[:10])) / len(relevant),
            "Recall@100": len(relevant & set(ranking[:100])) / len(relevant),
            "nDCG@10": sum(g / math.log2(i + 2) for i, g in enumerate(gains)) / idcg if idcg else 0.0,
            "MRR@10": 1 / (first_hit + 1) if first_hit is not None else 0.0,
        })
    return pd.DataFrame(rows)


def bootstrap_ci(values, n=1000):
    rng = np.random.default_rng(SEED)
    means = values[rng.integers(0, len(values), size=(n, len(values)))].mean(axis=1)
    return np.percentile(means, [2.5, 97.5])


results, rankings, timings = {}, {}, {}


def record(name, ranked_ids, seconds):
    rankings[name] = ranked_ids
    results[name] = per_query_metrics(ranked_ids)
    timings[name] = seconds
    summary = results[name].mean()
    print(f"{name:<44} " + " · ".join(f"{k} {v:.4f}" for k, v in summary.items()) + f"  ({seconds / 60:.1f} min)")


def evaluate_colbert(name, encoder, index):
    start = time.time()
    _, positions = index.search(encode_queries(encoder, eval_query_texts), k=TOP_K)
    record(name, [[index.doc_ids[p] for p in row] for row in positions.tolist()], time.time() - start)


print(f"{len(eval_keys):,} Test-Queries · Katalog {len(catalog):,} Produkte · "
      f"Ø {np.mean([sum(l in RELEVANT_LABELS for l in j.values()) for j in eval_judged]):.1f} Exact-Produkte je Query")

In [ ]:
import bm25s
import Stemmer

with timed("BM25"):
    stemmer = Stemmer.Stemmer("english")
    start = time.time()
    bm25 = bm25s.BM25()
    bm25.index(bm25s.tokenize(catalog["text"].tolist(), stopwords="english", stemmer=stemmer, show_progress=False),
               show_progress=False)
    bm25_ids, _ = bm25.retrieve(bm25s.tokenize(eval_query_texts, stopwords="english", stemmer=stemmer,
                                               show_progress=False),
                                k=TOP_K, show_progress=False, n_threads=os.cpu_count() or 1)
    catalog_index = catalog.index.to_numpy()
    record("BM25", [list(catalog_index[row]) for row in bm25_ids], time.time() - start)
    del bm25
    _ = gc.collect()

In [ ]:
BASE_NAME = "mLateOn-unsupervised (Basis)"
PUBLISHED_NAME = "mlateon-esci-product-search (veröffentlicht)"
TRAINED_NAME = f"Eigenes Training (Profil {PROFILE})"

if DENSE_MODEL:
    with timed("Dense"):
        start = time.time()
        dense = SentenceTransformer(DENSE_MODEL, device="cuda", model_kwargs={"dtype": AMP_DTYPE})
        dense.max_seq_length = DOCUMENT_TOKENS
        with torch.inference_mode():
            doc_vectors = dense.encode_document(catalog["text"].tolist(), batch_size=ENCODE_BATCH_SIZE,
                                                convert_to_tensor=True, normalize_embeddings=True,
                                                show_progress_bar=True)
            query_vectors = dense.encode_query(eval_query_texts, batch_size=ENCODE_BATCH_SIZE,
                                               convert_to_tensor=True, normalize_embeddings=True)
            top = torch.cat([(q @ doc_vectors.T).topk(TOP_K, dim=1).indices.cpu() for q in query_vectors.split(1024)])
        record(f"Dense ({DENSE_MODEL.split('/')[-1]})", [list(catalog_index[row]) for row in top.numpy()],
               time.time() - start)
        del dense, doc_vectors, query_vectors
        _ = gc.collect()
        torch.cuda.empty_cache()

with timed("Suche Basismodell"):
    evaluate_colbert(BASE_NAME, base_encoder, base_index)

if REFERENCE_MODEL:
    with timed("Index + Suche Referenz"):
        reference_revision = HfApi().model_info(REFERENCE_MODEL).sha
        reference_encoder = load_encoder(REFERENCE_MODEL, AMP_DTYPE, reference_revision).eval()
        reference_index = TokenIndex.build(
            reference_encoder, catalog.index, catalog["text"].to_numpy(),
            INDEX_DIR / f"reference-{catalog_fingerprint}",
            fingerprint=f"{REFERENCE_MODEL}@{reference_revision}|q{QUERY_TOKENS}|d{DOCUMENT_TOKENS}|{AMP_DTYPE}",
        )
        evaluate_colbert(f"{REFERENCE_MODEL.split('/')[-1]} (LightOn)", reference_encoder, reference_index)
        shutil.rmtree(reference_index.directory)   # nur für die Evaluation gebraucht; spart Platte
        del reference_encoder, reference_index
        _ = gc.collect()
        torch.cuda.empty_cache()

if PUBLISHED_MODEL:
    with timed("Index + Suche veröffentlichtes Modell"):
        published_encoder = load_encoder(PUBLISHED_MODEL, AMP_DTYPE, PUBLISHED_REVISION).eval()
        published_index = TokenIndex.build(
            published_encoder, catalog.index, catalog["text"].to_numpy(),
            INDEX_DIR / f"published-{catalog_fingerprint}",
            fingerprint=f"{PUBLISHED_MODEL}@{PUBLISHED_REVISION}|q{QUERY_TOKENS}|d{DOCUMENT_TOKENS}|{AMP_DTYPE}",
        )
        evaluate_colbert(PUBLISHED_NAME, published_encoder, published_index)

with timed("Index + Suche eigenes Modell"):
    weights = MODEL_DIR / "model.safetensors"
    trained_index = TokenIndex.build(
        trained_encoder, catalog.index, catalog["text"].to_numpy(),
        INDEX_DIR / f"{RUN_NAME}-{catalog_fingerprint}",
        fingerprint=f"{MODEL_DIR}|{weights.stat().st_size}|{weights.stat().st_mtime_ns}|{AMP_DTYPE}",
    )
    evaluate_colbert(TRAINED_NAME, trained_encoder, trained_index)

In [ ]:
table = []
for name, frame in results.items():
    row = {"Verfahren": name}
    for metric in frame.columns:
        low, high = bootstrap_ci(frame[metric].to_numpy())
        row[metric] = frame[metric].mean()
        row[f"{metric} 95%-KI"] = f"{low:.3f}–{high:.3f}"
    table.append(row)
table = pd.DataFrame(table).set_index("Verfahren")
(RUN_DIR / "test_metrics.json").write_text(json.dumps(
    {"profile": PROFILE, "catalog_products": len(catalog), "test_queries": len(eval_keys),
     "metrics": json.loads(table.to_json(orient="index"))}, indent=2))
display(Markdown(f"**{len(eval_keys):,} Test-Queries, exakt über {len(catalog):,} Produkte"
                 + (" (Teilkatalog – nicht 1:1 mit dem vollen Katalog vergleichbar)**"
                    if len(catalog) < len(all_product_ids) else "**")))
display(table[["Recall@10", "Recall@100", "nDCG@10", "MRR@10", "Recall@100 95%-KI", "nDCG@10 95%-KI"]]
        .style.format({m: "{:.3f}" for m in ["Recall@10", "Recall@100", "nDCG@10", "MRR@10"]}))

delta = results[TRAINED_NAME] - results[BASE_NAME]
paired = [(TRAINED_NAME, BASE_NAME)] + ([(PUBLISHED_NAME, TRAINED_NAME)] if PUBLISHED_MODEL else [])
for a, b in paired:
    for metric in ["Recall@100", "nDCG@10"]:
        d = (results[a] - results[b])[metric].to_numpy()
        low, high = bootstrap_ci(d)
        print(f"{metric}: {a} vs. {b} {d.mean():+.3f} (95%-KI {low:+.3f} bis {high:+.3f}), "
              f"besser bei {(d > 0).mean():.0%}, schlechter bei {(d < 0).mean():.0%} der Queries")

fig, ax = plt.subplots(figsize=(9, 0.5 + 0.45 * len(table)))
ax.barh(table.index[::-1], table["Recall@100"][::-1], color="#5b8def")
for y, value in enumerate(table["Recall@100"][::-1]):
    ax.text(value + 0.005, y, f"{value:.1%}", va="center")
ax.set(xlabel="Recall@100 – Anteil der Exact-Produkte in den Top 100", xlim=(0, 1),
       title=f"{len(eval_keys):,} Test-Queries · {len(catalog):,} Produkte im Katalog")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## 9b · LLM-Judge für unbeurteilte Treffer

ESCI beurteilt je Query nur die Produkte, die Annotatoren damals gesehen haben – im Median 16. Bei Suche über einen
großen Katalog ist der Großteil der Top-10 unbeurteilt und zählt oben als Fehltreffer. Der Datensatz
[`johannhartmann/esci-llm-judged-top10`](https://huggingface.co/datasets/johannhartmann/esci-llm-judged-top10)
schließt die Lücken für 1.000 Test-Queries: Claude Haiku 4.5 hat jeden unbeurteilten Top-10-Treffer von BM25,
Basismodell und veröffentlichtem Modell **über den vollen Katalog** nach den ESCI-Definitionen gelabelt,
kalibriert an 1.500 blind mitgelabelten menschlichen Urteilen. Menschliche Labels haben immer Vorrang.

Standard: Urteile vom Hub laden, keine API-Kosten. Zwei Auswertungen:
1. **Voller Katalog (Referenz):** die veröffentlichten Top-10-Listen, neu berechnet aus Rankings und Labels.
2. **Dieser Lauf:** die Rankings aus Abschnitt 9 für dieselben Queries. Treffer, die im veröffentlichten Pool
   fehlen – etwa weil das eigene Modell anders rankt –, bleiben unbeurteilt; `unlabeled@10` zeigt, wie viele.

nDCG@10 nutzt als ideale Liste alle bekannten Labels der Query; Werte sind nur innerhalb eines Modus vergleichbar.

In [ ]:
# @title Urteile vom Hub laden, Kalibrierung ansehen
judge_data = {name: pd.read_parquet(judge_file(f"data/{name}.parquet"))
              for name in ["queries", "judgments", "calibration", "qrels"]}
pool_table = judge_data["judgments"]
print(f"{len(judge_data['queries']):,} Queries · Pool {len(pool_table):,} Paare "
      f"(menschlich {pool_table['label_source'].eq('human').sum():,}, "
      f"LLM {pool_table['label_source'].eq('llm').sum():,}) · Judge {ej.JUDGE_MODEL}")

calibration = judge_data["calibration"].dropna(subset=["judge_label"])
calibration_rows = []
for part, frame in [("alle", calibration), *calibration.groupby("stratum")]:
    report = ej.calibration_report(frame["human_label"], frame["judge_label"])
    calibration_rows.append({
        "Teilmenge": {"pooled": "aus dem Top-10-Pool", "not_pooled": "übrige ESCI-Urteile"}.get(part, part),
        "Paare": report["n"],
        "Genauigkeit (E/S/C/I)": report["accuracy"], "κ (E/S/C/I)": report["kappa"],
        "Genauigkeit E vs. Rest": report["E_vs_rest"]["accuracy"], "κ E vs. Rest": report["E_vs_rest"]["kappa"],
        "Genauigkeit E+S vs. C+I": report["ES_vs_CI"]["accuracy"],
    })
display(Markdown("**Kalibrierung:** Der Judge labelt menschlich beurteilte Paare derselben Queries, ohne das "
                 "menschliche Label zu sehen."))
display(pd.DataFrame(calibration_rows).set_index("Teilmenge").style.format(precision=3))
names = [ej.LABEL_NAMES[l] for l in ej.LABELS]
confusion = pd.crosstab(calibration["human_label"].map(ej.LABEL_NAMES).rename("Mensch (ESCI)"),
                        calibration["judge_label"].map(ej.LABEL_NAMES).rename("LLM-Judge"))
display(confusion.reindex(index=names, columns=names, fill_value=0))

qrels_by_query = {}
for r in judge_data["qrels"].itertuples():
    qrels_by_query.setdefault(r.query_key, {})[r.product_id] = (r.label, r.label_source)
llm_labels = {(k, pid): label for k, labels in qrels_by_query.items()
              for pid, (label, source) in labels.items() if source == "llm"}
JUDGE_METRICS = ["P@10 (E)", "P@10 (E+S)", "nDCG@10", "unlabeled@10"]


def show_judge_table(rows, order, title):
    frame = pd.DataFrame(rows)
    shown = frame.set_index(["method", "mode"])[JUDGE_METRICS].unstack("mode").reindex(order)
    shown.columns = [f"{m} · {mode}" for m, mode in shown.columns]
    display(Markdown(title))
    display(shown.style.format("{:.3f}"))
    return frame

In [ ]:
# @title Referenz: veröffentlichtes Modell über den vollen Katalog (aus dem Hub neu berechnet)
top10 = json.loads(Path(judge_file("raw/top10.json")).read_text())
reference_keys = top10["sample"]
reference_rankings = {"BM25": top10["top10"]["bm25"], BASE_NAME: top10["top10"]["base"],
                      PUBLISHED_NAME: top10["top10"]["finetuned"]}
reference_modes = {
    "ESCI-only": {k: {p: l for p, (l, s) in qrels_by_query[k].items() if s == "human"} for k in reference_keys},
    "LLM-vervollständigt": {k: {p: l for p, (l, s) in qrels_by_query[k].items()} for k in reference_keys},
}
with timed("LLM-Metriken"):
    reference_rows, reference_per_query = ej.metrics_table(reference_rankings, reference_keys, reference_modes)
reference_table = show_judge_table(
    reference_rows, list(reference_rankings),
    f"**Voller Katalog (482.105 Produkte), {len(reference_keys):,} Test-Queries, Top-10.** "
    "ESCI-only: unbeurteilt = irrelevant. LLM-vervollständigt: menschliches Label, sonst Judge-Label.")

published_results = json.loads(Path(judge_file("results.json")).read_text())
hub_values = {(r["mode"].replace("LLM-completed", "LLM-vervollständigt"), r["method"].split(" (")[0]): r["nDCG@10"]
              for r in published_results["metrics"]}
method_hub_name = {"BM25": "BM25", BASE_NAME: "mLateOn-unsupervised", PUBLISHED_NAME: "mlateon-esci-product-search"}
gap = max(abs(r["nDCG@10"] - hub_values[(r["mode"], method_hub_name[r["method"]])]) for r in reference_rows)
print(f"Abgleich mit results.json des Datensatzes: größte Abweichung nDCG@10 = {gap:.1e}")
for mode in reference_modes:
    d = ej.paired_delta(reference_per_query, mode, PUBLISHED_NAME, BASE_NAME, "nDCG@10")
    print(f"{mode:<19} nDCG@10 veröffentlicht vs. Basis {d['delta']:+.3f} (95-%-KI {d['lo']:+.3f} bis {d['hi']:+.3f})")

### Optional: Urteile für die eigenen Treffer erzeugen

`REGENERATE_JUDGMENTS = True` lässt die unbeurteilten Top-10-Treffer **dieses Laufs** (nicht im Hub-Pool) für die
ersten `REGEN_QUERIES` Judge-Queries von Claude Haiku 4.5 labeln – dieselbe Pipeline wie im Datensatz (distilabel,
JSON-Schema, gecachter Prompt, Temperatur 0), dazu blind die Kalibrierpaare dieser Queries. Voraussetzungen:
Colab-Secret `ANTHROPIC_API_KEY`; `JUDGE_BUDGET_USD` ist eine harte Obergrenze, geprüft vor jeder Anfrage.
Vorher wird der Antwort-Cache vom Hub eingespielt. Ein Upload der bezahlten Antworten erfolgt nur mit
`HUB_SYNC_REPO` (eigenes Datensatz-Repo, Schreibrecht per `HF_TOKEN`).

In [ ]:
# @title Optional: Urteile selbst erzeugen (distilabel + Claude Haiku 4.5, gecacht, budgetiert)
REGENERATE_JUDGMENTS = False  # @param {type:"boolean"}
REGEN_QUERIES = 20            # @param {type:"integer"}   0 = alle Judge-Queries dieses Laufs
JUDGE_BUDGET_USD = 1.0        # @param {type:"number"}    harte Obergrenze für neue API-Kosten
HUB_SYNC_REPO = ""            # @param {type:"string"}    eigenes Datensatz-Repo für die bezahlten Antworten

eval_position = {k: i for i, k in enumerate(eval_keys)}
judge_keys = [k for k in judge_data["queries"]["query_key"] if k in eval_position]

if REGENERATE_JUDGMENTS:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "distilabel==1.5.3", "anthropic>=1.8"])
    judge_file("code/esci_judge_llm.py")          # liegt im selben Snapshot-Ordner wie esci_judge.py
    # `import distilabel` installiert rich-Tracebacks mit show_locals=True; lokale Variablen können Keys enthalten.
    # Deshalb die bisherigen Handler (sys.excepthook, IPython-Traceback-Anzeige) sofort wiederherstellen.
    from IPython import get_ipython
    shell = get_ipython()
    saved_hook, saved_show = sys.excepthook, getattr(shell, "_showtraceback", None)
    import distilabel  # noqa: F401
    sys.excepthook = saved_hook
    if saved_show is not None:
        shell._showtraceback = saved_show
    from esci_judge_llm import run_judge
    api_key = colab_secret("ANTHROPIC_API_KEY")
    if api_key:
        os.environ["ANTHROPIC_API_KEY"] = api_key
    JUDGE_DIR = LOCAL_ROOT / "llm_judge"
    JUDGE_DIR.mkdir(parents=True, exist_ok=True)

    regen_keys = judge_keys[:REGEN_QUERIES or None]
    to_judge = {}
    for name, ranking in rankings.items():
        for key in regen_keys:
            for pid in ranking[eval_position[key]][:10]:
                if pid not in judged[key] and (key, pid) not in llm_labels:
                    to_judge.setdefault(key, []).append(pid)
    calibration_pairs = judge_data["calibration"][judge_data["calibration"]["query_key"].isin(regen_keys)]
    for row in calibration_pairs.itertuples():
        to_judge.setdefault(row.query_key, []).append(row.product_id)
    fields = ej.read_product_fields(products_path, {p for ps in to_judge.values() for p in ps})
    judge_texts = {pid: ej.judge_product_text(row) for pid, row in fields.to_dict("index").items()}
    groups = ej.make_groups(to_judge, queries.to_dict(), judge_texts)
    print("Antwort-Cache vom Hub – neue Einträge:", ej.restore_caches(JUDGE_DATASET, JUDGE_DIR, HF_TOKEN))
    uncached, estimate = ej.estimate_cost(groups, JUDGE_DIR / "llm_cache.sqlite")
    print(f"{len(regen_keys)} Queries, {sum(len(g['product_ids']) for g in groups):,} Paare in {len(groups)} "
          f"Anfragen; nicht im Cache: {uncached} (geschätzt ${estimate:.2f}, Obergrenze ${JUDGE_BUDGET_USD:.2f})")
    if uncached and not os.environ.get("ANTHROPIC_API_KEY"):
        raise RuntimeError("ANTHROPIC_API_KEY fehlt: Colab-Secret anlegen und für dieses Notebook freigeben.")
    judge_start = time.time()
    try:
        with timed("LLM-Judge (API)"):
            regenerated, judge_cost = run_judge(groups, JUDGE_DIR, budget_usd=JUDGE_BUDGET_USD,
                                                total_cap_usd=JUDGE_BUDGET_USD, name="esci-llm-judge-colab")
    finally:
        if HUB_SYNC_REPO and ej.ledger_cost(JUDGE_DIR / "api_ledger.jsonl", judge_start) > 0:
            url = ej.push_caches(HUB_SYNC_REPO, JUDGE_DIR, colab_secret("HF_TOKEN"), "colab: judge responses")
            print("Bezahlte Antworten hochgeladen:" if url else f"Kein Schreibrecht für {HUB_SYNC_REPO}.", url or "")
    calibration_check = [(r.human_label, regenerated[(r.query_key, r.product_id)]["label"])
                         for r in calibration_pairs.itertuples() if (r.query_key, r.product_id) in regenerated]
    if calibration_check:
        agreement = np.mean([h == j for h, j in calibration_check])
        print(f"Kalibrierung dieses Laufs: {agreement:.0%} Übereinstimmung mit ESCI ({len(calibration_check)} Paare)")
    new = 0
    for (key, pid), j in regenerated.items():
        if pid not in judged[key] and (key, pid) not in llm_labels:
            llm_labels[(key, pid)] = j["label"]
            new += 1
    print(f"Neue API-Kosten: ${judge_cost:.2f} · {new:,} neue Labels für Treffer dieses Laufs")

In [ ]:
# @title Dieser Lauf: ESCI-only vs. LLM-vervollständigt
human_qrels = {k: judged[k] for k in judge_keys}
completed_qrels = {k: dict(judged[k]) for k in judge_keys}
for (key, pid), label in llm_labels.items():
    if key in completed_qrels:
        completed_qrels[key].setdefault(pid, label)   # menschliche Labels haben Vorrang
judge_rankings = {name: {k: list(ranking[eval_position[k]][:10]) for k in judge_keys}
                  for name, ranking in rankings.items()}
MODES = {"ESCI-only": human_qrels, "LLM-vervollständigt": completed_qrels}
with timed("LLM-Metriken"):
    judge_rows, judge_per_query = ej.metrics_table(judge_rankings, judge_keys, MODES)
judge_table = show_judge_table(
    judge_rows, list(rankings),
    f"**Dieser Lauf: {len(judge_keys):,} Test-Queries, Top-10 über {len(catalog):,} Produkte.** "
    "`unlabeled@10` im vervollständigten Modus: Treffer außerhalb des gelabelten Pools.")
judge_deltas = [ej.paired_delta(judge_per_query, mode, TRAINED_NAME, BASE_NAME, metric)
                for mode in MODES for metric in JUDGE_METRICS[:3]]
for d in judge_deltas:
    print(f"{d['mode']:<19} {d['metric']:<11} eigenes Training vs. Basis {d['delta']:+.3f} "
          f"(95-%-KI {d['lo']:+.3f} bis {d['hi']:+.3f}), besser bei {d['better']:.0%}, schlechter bei {d['worse']:.0%}")
(RUN_DIR / "test_metrics_llm_completed.json").write_text(json.dumps(
    {"judge_dataset": JUDGE_DATASET, "judge_revision": JUDGE_REVISION, "catalog_products": len(catalog),
     "queries": len(judge_keys), "metrics": judge_rows, "paired_trained_vs_base": judge_deltas,
     "full_catalog_reference": reference_rows, "regenerated": bool(REGENERATE_JUDGMENTS)}, indent=2))

fig, ax = plt.subplots(figsize=(9, 0.6 + 0.55 * len(rankings)))
names = list(rankings)[::-1]
y = np.arange(len(names))
for offset, (mode, color) in zip((-0.2, 0.2), (("ESCI-only", "#b0b8c8"), ("LLM-vervollständigt", "#5b8def"))):
    values = judge_table[judge_table["mode"] == mode].set_index("method").loc[names, "nDCG@10"]
    ax.barh(y + offset, values, height=0.38, color=color, label=mode)
    for yy, v in zip(y + offset, values):
        ax.text(v + 0.005, yy, f"{v:.3f}", va="center", fontsize=8)
ax.set(yticks=y, yticklabels=names, xlabel="nDCG@10 (ESCI-Gains)", xlim=(0, 1),
       title=f"{len(judge_keys):,} Test-Queries, {len(catalog):,} Produkte: unbeurteilt = irrelevant vs. LLM-vervollständigt")
ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## 10 · Suche: vorher und nachher

Dieselbe Query, derselbe Katalog: Basismodell, eigenes Modell und – falls geladen – das veröffentlichte Vollmodell.
Bei Test-Queries steht neben jedem Treffer das ESCI-Urteil (E/S/C/I, `–` = nicht beurteilt; heißt nicht
„irrelevant“). Jede Suche ist exakt über den ganzen Katalog.

In [ ]:
SEARCH_MODELS = [("Basis", base_encoder, base_index), ("Eigenes", trained_encoder, trained_index)]
if PUBLISHED_MODEL:
    SEARCH_MODELS.append(("Veröffentlicht", published_encoder, published_index))


def compare(query_list, k=10):
    """Top-k aller Modelle für eine Liste von Queries (ein Durchlauf über jeden Index)."""
    hits = {}
    for label, encoder, index in SEARCH_MODELS:
        _, positions = index.search(encode_queries(encoder, query_list), k=k, progress=False)
        hits[label] = [[index.doc_ids[p] for p in row] for row in positions.tolist()]
    tables = []
    for i, query in enumerate(query_list):
        labels = judged.get(normalize_query([query]).iloc[0], {})
        table = {"Rang": list(range(1, k + 1))}
        for label, ids in hits.items():
            table[f"{label}: Produkt"] = [catalog.at[d, "title"][:70] for d in ids[i]]
            table[f"{label}: Urteil"] = [labels.get(d, "–") for d in ids[i]]
        tables.append(pd.DataFrame(table).set_index("Rang"))
    return tables


def search(query, k=10):
    start = time.time()
    frame = compare([query], k)[0]
    print(f"{query!r}: {time.time() - start:.1f} s über {len(catalog):,} Produkte")
    display(frame)


candidates = [i for i, k in enumerate(eval_keys) if presentable(queries[k])]
by_delta = sorted(candidates, key=lambda i: delta["nDCG@10"].iloc[i])
pools = {"größte Verbesserung": by_delta[::-1][:20], "größte Verschlechterung": by_delta[:20],
         "zufällig": random.Random(SEED).sample(candidates, 20)}
pool_queries = list(dict.fromkeys(queries[eval_keys[i]] for pool in pools.values() for i in pool))
with timed("Vorher/Nachher-Suche"):
    start = time.time()
    pool_tables = dict(zip(pool_queries, compare(pool_queries)))
print(f"{len(pool_queries)} Queries × {len(SEARCH_MODELS)} Modelle, exakt über {len(catalog):,} Produkte: "
      f"{time.time() - start:.1f} s")

example_queries = []
for why, pool in pools.items():
    i = next(i for i in pool
             if presentable(*pool_tables[queries[eval_keys[i]]].filter(like="Produkt").to_numpy().ravel()))
    query = queries[eval_keys[i]]
    example_queries.append(query)
    display(Markdown(
        f"**{query}** – {why} · nDCG@10 {results[BASE_NAME]['nDCG@10'].iloc[i]:.2f} → "
        f"{results[TRAINED_NAME]['nDCG@10'].iloc[i]:.2f} · Recall@100 "
        f"{results[BASE_NAME]['Recall@100'].iloc[i]:.2f} → {results[TRAINED_NAME]['Recall@100'].iloc[i]:.2f}"))
    display(pool_tables[query])

In [ ]:
# @title Eigene Suche (Widget)
import ipywidgets as widgets

query_box = widgets.Combobox(
    value=example_queries[-1],
    options=[queries[eval_keys[i]] for i in candidates[:500]],
    placeholder="Suchtext oder Test-Query",
    description="Suche:",
    ensure_option=False,
    layout=widgets.Layout(width="80%"),
)
search_button = widgets.Button(description="Vergleichen", button_style="primary")
search_output = widgets.Output()


def on_search(_):
    with search_output:
        search_output.clear_output(wait=True)
        if query_box.value.strip():
            search(query_box.value.strip())
        else:
            print("Bitte einen Suchtext eingeben.")


search_button.on_click(on_search)
display(widgets.VBox([widgets.HBox([query_box, search_button]), search_output]))

In [ ]:
# @title Eigene Suche (ohne Widget)
SEARCH_QUERY = "wireless mouse for travel"  # @param {type:"string"}
search(SEARCH_QUERY)

## 11 · Export

Der Modellordner (~1,2 GB) ist vollständig und direkt ladbar.

- **Hugging Face Hub:** `HUB_REPO_ID` setzen (z. B. `"mein-name/mlateon-esci"`) und in den Colab-Secrets
  (Schlüssel-Symbol links) `HF_TOKEN` mit Schreibrecht anlegen und für das Notebook freigeben. Das Repository wird
  standardmäßig privat angelegt; hochgeladen wird der gespeicherte Ordner mit Modellkarte und `training_recipe.json`.
- **Google Drive:** `COPY_TO_DRIVE=True` kopiert Modell und Ergebnisse nach `MyDrive/colbert_training/models/`.
  Mit `USE_GOOGLE_DRIVE=True` liegen sie schon dort.

In [ ]:
# @title Export
HUB_REPO_ID = ""        # @param {type:"string"}
HUB_PRIVATE = True      # @param {type:"boolean"}
COPY_TO_DRIVE = False   # @param {type:"boolean"}

if HUB_REPO_ID:
    token = colab_secret("HF_TOKEN")
    if not token:
        print("Kein HF_TOKEN (Umgebung oder Colab-Secret) – Upload übersprungen.")
    else:
        commit = trained_encoder.push_to_hub(HUB_REPO_ID, token=token, private=HUB_PRIVATE, exist_ok=True,
                                             local_model_path=str(MODEL_DIR),
                                             commit_message=f"{RUN_NAME}: {recipe['optimizer_steps']} steps")
        print("Hochgeladen:", commit)

if COPY_TO_DRIVE:
    drive_root = mount_drive()
    if drive_root is None:
        print("COPY_TO_DRIVE wirkt nur in Colab; das Modell liegt unter", MODEL_DIR)
    elif MODEL_DIR.is_relative_to(drive_root):
        print("Modell liegt bereits auf Drive:", MODEL_DIR)
    else:
        target = drive_root / "colbert_training" / "models" / RUN_NAME
        shutil.copytree(MODEL_DIR, target / "model", dirs_exist_ok=True)
        for name in ["test_metrics.json", "test_metrics_llm_completed.json", "training_log.json"]:
            if (RUN_DIR / name).exists():
                shutil.copy2(RUN_DIR / name, target / name)
        print("Nach Drive kopiert:", target)

print("Modellordner:", MODEL_DIR)
for path in sorted(MODEL_DIR.iterdir()):
    print(f"  {path.name}")

In [ ]:
# @title Laufzeit und Ressourcen
def folder_gib(path):
    return sum(p.stat().st_size for p in Path(path).rglob("*") if p.is_file()) / 2**30 if Path(path).exists() else 0.0


resources = {
    "profile": PROFILE, "gpu": GPU_NAME, "mixed_precision": "bf16" if BF16 else "fp16",
    "catalog_products": len(catalog), "test_queries": len(eval_keys),
    "minutes": {phase: round(minutes, 1) for phase, minutes in TIMINGS.items()},
    "gpu_peak_allocated_gib": round(max(PEAK_BEFORE_TRAINING_GIB, torch.cuda.max_memory_allocated() / 2**30), 2),
    "training_peak_allocated_gib": recipe["training_peak_allocated_gib"],
    "ram_peak_rss_gib": round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20, 2),
    "disk_gib": {"data": round(folder_gib(DATA_DIR), 1), "index": round(folder_gib(INDEX_DIR), 1),
                 "run (Tripel, Checkpoints, Modell)": round(folder_gib(RUN_DIR), 1)},
}
(RUN_DIR / "resources.json").write_text(json.dumps(resources, indent=2))
display(pd.Series(resources["minutes"], name="Minuten").to_frame())
print(f"Gesamt {sum(TIMINGS.values()):.0f} min · GPU-Spitze (PyTorch) {resources['gpu_peak_allocated_gib']:.1f} GiB · "
      f"RAM-Spitze (RSS inkl. gelesener Index-Seiten) {resources['ram_peak_rss_gib']:.1f} GiB · "
      f"Platte {sum(resources['disk_gib'].values()):.1f} GiB {resources['disk_gib']}")

### Das Modell woanders verwenden

```python
from sentence_transformers import MultiVectorEncoder

model = MultiVectorEncoder("pfad/zum/model")        # oder die Hub-ID
queries = model.encode_query(["wireless mouse for travel"])
products = model.encode_document(["Compact wireless mouse with USB receiver", "Mouse pad, extra large"])
print(model.similarity(queries, products))          # MaxSim-Scores
```

Für viele Suchanfragen pro Sekunde: Dokumentvektoren einmal berechnen und in einen Multi-Vector-Index legen
(PLAID/[FastPlaid](https://github.com/lightonai/fast-plaid)) oder die `TokenIndex`-Klasse oben für exakte Batch-Suche.

### Quellen
- [Amazon ESCI – Daten und Lizenz](https://github.com/amazon-science/esci-data) · [Paper](https://arxiv.org/abs/2206.06588)
- [mLateOn-unsupervised](https://huggingface.co/lightonai/mLateOn-unsupervised) · [mlateon-esci-product-search](https://huggingface.co/johannhartmann/mlateon-esci-product-search) · [esci-llm-judged-top10](https://huggingface.co/datasets/johannhartmann/esci-llm-judged-top10)
- [Multi-Vector-Training mit Sentence Transformers](https://huggingface.co/blog/train-multi-vector-encoder)
- [NV-Retriever: Hard-Negative-Mining mit Positiv-relativer Marge](https://huggingface.co/papers/2407.15831)
- [bm25s](https://github.com/xhluca/bm25s)